In [8]:
!pip install clustering-benchmarks -q

In [9]:
import clustbench
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"

In [10]:
battery_datasets_dict = {
                         'fcps': ['atom', 'chainlink', 'engytime', 'hepta', 'lsun', 'target', 'tetra', 'twodiamonds', 'wingnut'],
                         'uci': ['ecoli', 'glass', 'ionosphere', 'sonar', 'statlog', 'wdbc', 'wine', 'yeast'],
                         #'mnist': ['digits', 'fashion'], # Genie digits ~ 32 min
                         #'sipu': ['worms_64'], # Genie ~ 9 min
                         }

| Battery | Dataset    | Rows   | Cols   | Clusters
|---------|------------|--------|--------|----------
| uci     | ecoli      | 336    | 7      | 8
| uci     | glass      | 214    | 9      | 6
| uci     | ionosphere | 351    | 33     | 2
| uci     | sonar      | 208    | 60     | 2
| uci     | statlog    | 2310   | 18     | 7
| uci     | wdbc       | 569    | 30     | 2
| uci     | wine       | 178    | 13     | 3
| uci     | yeast      | 1484   | 8      | 4
| mnist   | digits     | 70000  | 719    | 10
| mnist   | fashion    | 70000  | 784    | 10
| sipu    | worms_64   | 105000 | 64     | 25


In [11]:
import numpy as np
from sklearn.neural_network import MLPRegressor
import pandas as pd

def ssnn_feat_eng(X, n_embeddings=2):

  # To store embeddings for each feature
  embeddings_list = []

  # Get the number of features
  n_samples, n_features = X.shape

  for feature_idx in range(n_features):

      # Prepare input by removing the current feature
      X_in = np.delete(X, feature_idx, axis=1)  # shape: (n_samples, n_features-1)

      # Target is the left-out feature
      y_target = X[:, feature_idx]  # shape: (n_samples,)

      # Define MLP
      mlp = MLPRegressor(
          hidden_layer_sizes=(64, n_embeddings),
          # Use 'identity' for linear activation in the hidden layers
          activation='logistic', #{'relu', 'logistic', 'identity', 'tanh'}
          max_iter=10000,
          random_state=42
      )

      # Fit the model
      mlp.fit(X_in, y_target)

      # Extract learned weights and biases
      coefs = mlp.coefs_
      intercepts = mlp.intercepts_

      # Forward pass to hidden layer 1 (linear)
      Z1 = X_in @ coefs[0] + intercepts[0]  # shape: (n_samples, 64)

      # Forward pass to hidden layer 2 (linear) → this is the embedding
      embeddings = Z1 @ coefs[1] + intercepts[1]  # shape: (n_samples, n_embeddings)

      # Append embeddings
      embeddings_list.append(embeddings)

  # Concatenate all embeddings horizontally
  X_new = np.hstack(embeddings_list)  # shape: (n_samples, n_embeddings * n_features)
  X_new_df = pd.DataFrame(X_new, columns=[f'feature_{i+1}_emb_{j+1}' for i in range(n_features) for j in range(2)])
  return X_new

In [12]:
data_url = "https://github.com/gagolews/clustering-data-v1/raw/v1.1.0"
battery = "uci"
dataset = "statlog"
b = clustbench.load_dataset(battery, dataset, url=data_url)

In [13]:
pd.DataFrame(b.data)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,1.765322,1.035127,0.001837,-0.000088,-0.020114,-0.097886,-0.024911,-0.146012,0.428179,0.372141,0.588547,0.323848,-0.168113,0.481106,-0.312992,0.570538,-0.002047,-0.012852
1,-0.225939,0.124837,-0.000272,-0.000089,-0.030650,-0.103515,-0.039663,-0.149412,-0.685804,-0.622436,-0.789531,-0.645442,0.190103,-0.311184,0.121081,-0.807540,0.010869,-0.014419
2,1.461892,-1.562994,-0.000271,-0.000089,-0.018007,-0.093628,-0.024913,-0.136887,1.630661,1.499467,1.812804,1.579710,-0.393578,0.546428,-0.152850,1.794796,-0.004316,-0.017770
3,-1.762054,0.940305,-0.000272,-0.000090,-0.003257,-0.074487,0.124696,-0.028337,0.124045,0.127710,0.165010,0.079417,0.010995,0.122891,-0.133883,0.147001,-0.003033,-0.012061
4,-1.212087,1.395451,-0.000271,-0.000090,-0.008524,-0.079536,0.003534,-0.119822,0.237831,0.216210,0.329368,0.167918,-0.064862,0.274606,-0.209742,0.311358,-0.002351,-0.012506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2305,-1.799982,-0.406166,-0.000273,-0.000090,-0.012739,-0.106027,-0.020699,-0.141164,-0.318457,-0.236828,-0.363886,-0.354656,0.244889,-0.136289,-0.108598,-0.381894,-0.000869,-0.003645
2306,0.342993,-1.885388,-0.000272,-0.000090,-0.011686,-0.091065,-0.029128,-0.134702,1.717758,1.609039,1.848626,1.695604,-0.326151,0.392605,-0.066455,1.830617,-0.004882,-0.018706
2307,-0.851764,-0.975098,-0.000271,-0.000090,-0.012739,-0.089239,-0.018590,-0.134197,0.416238,0.351069,0.573797,0.323849,-0.195506,0.472677,-0.277171,0.555789,-0.002129,-0.013793
2308,-0.510405,0.181730,-0.000272,-0.000089,-0.025381,-0.105010,-0.038606,-0.150121,-0.684399,-0.622436,-0.785317,-0.645444,0.185890,-0.302755,0.116867,-0.803327,0.010869,-0.014420


In [14]:
autoencoder_feat_eng(b.data)

array([[ 13.81004197, -14.82063806,  -1.39051573, ...,  -0.46980492,
          0.47607998,  -0.50768348],
       [-18.47161276,  18.31404658,  39.59451831, ...,  -0.78837141,
          1.05236506,  -1.09346264],
       [ 66.27576784, -69.36981416, -54.02396342, ...,  -0.20888969,
         -0.42412393,   0.77637578],
       ...,
       [ 11.72831274, -10.58978975, -11.76614186, ...,  -0.52122604,
         -0.17285507,   0.4541925 ],
       [-18.60586915,  18.43868262,  38.46313635, ...,  -0.78995395,
          1.05466575,  -1.09889246],
       [-19.9497942 ,  20.04319461,  30.72918317, ...,  -0.79849741,
          1.02356528,  -1.07582964]])

In [15]:
import genieclust
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores(battery, dataset, apply_scale=False):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = ssnn_feat_eng(b.data)
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

In [16]:
import genieclust
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_tsne(battery, dataset, apply_scale=False, pca_components=-1, tsne_components=2, tsne_perplexity=30):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = ssnn_feat_eng(b.data)
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  if pca_components != -1:
    pca = PCA(n_components=pca_components)
    X_transformed = pca.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  tsne = TSNE(n_components=tsne_components, perplexity=tsne_perplexity, random_state=42) # Reduce to 2 dimensions
  X_transformed = tsne.fit_transform(X_transformed)
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

In [17]:
import genieclust
from umap import UMAP
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_scores_with_umap(battery, dataset, apply_scale=False, pca_components=-1, umap_components=2, n_neighbors=15, min_dist=0.1):
  b = clustbench.load_dataset(battery, dataset, url=data_url)

  X_transformed = ssnn_feat_eng(b.data)
  if apply_scale:
    # Scale the data
    scaler = StandardScaler()
    X_transformed = scaler.fit_transform(X_transformed)

  if pca_components != -1:
    pca = PCA(n_components=pca_components)
    X_transformed = pca.fit_transform(X_transformed)

  g = genieclust.Genie(n_clusters=b.n_clusters[0])
  umap = UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist, random_state=42, n_jobs=1) # Reduce to 2 dimensions
  X_transformed = umap.fit_transform(X_transformed)
  results = clustbench.fit_predict_many(g, X_transformed, b.n_clusters)
  scores = clustbench.get_score(b.labels, results)

  return scores

In [18]:
import tqdm
import pandas as pd
columns = ['Battery', 'Dataset', 'Genie NCA Score']
df = pd.DataFrame(columns=columns)
scores_lists = {}
for col in columns:
  scores_lists[col] = []

for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Batteries"):
  for dataset in tqdm.tqdm(battery_datasets_dict[battery], desc=f"Processing Datasets in {battery}", leave=False):
    scores_lists['Battery'].append(battery)
    scores_lists['Dataset'].append(dataset)
    scores_lists['Genie NCA Score'].append(get_scores(battery, dataset))

df = pd.DataFrame.from_dict(scores_lists)

Processing Batteries: 100%|██████████| 2/2 [03:36<00:00, 108.34s/it]


In [19]:
df

,Battery,Dataset,Genie NCA Score
0,fcps,atom,0.977500
1,fcps,chainlink,1.000000
2,fcps,engytime,0.918870
3,fcps,hepta,1.000000
4,fcps,lsun,1.000000
5,fcps,target,1.000000
6,fcps,tetra,0.736667
7,fcps,twodiamonds,0.987500
8,fcps,wingnut,1.000000
9,uci,ecoli,0.405951


In [20]:
tsne_perplexity = [15,30,45]

scores_lists = {}
for perplexity in tqdm.tqdm(tsne_perplexity, desc="Processing TSNE Perplexity"):
  for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Batteries"):
    for dataset in tqdm.tqdm(battery_datasets_dict[battery], desc=f"Processing Datasets in {battery}", leave=False):
      column_name = 'Genie+TSNE'+ str(perplexity) +' NCA Score'
      if column_name not in scores_lists:
        scores_lists[column_name] = []
      scores_lists[column_name].append(get_scores_with_tsne(battery, dataset, tsne_perplexity=50))

df_tsne = pd.DataFrame.from_dict(scores_lists)

Processing Batteries:   0%|          | 0/2 [00:00<?, ?it/s]

Processing Datasets in fcps:   0%|          | 0/9 [00:00<?, ?it/s]

Processing Datasets in fcps:  11%|█         | 1/9 [00:11<01:29, 11.25s/it]

Processing Datasets in fcps:  22%|██▏       | 2/9 [00:23<01:22, 11.75s/it]

Processing Datasets in fcps:  33%|███▎      | 3/9 [01:21<03:18, 33.11s/it]

Processing Datasets in fcps:  44%|████▍     | 4/9 [01:25<01:46, 21.29s/it]

Processing Datasets in fcps:  56%|█████▌    | 5/9 [01:29<01:00, 15.15s/it]

Processing Datasets in fcps:  67%|██████▋   | 6/9 [01:38<00:39, 13.11s/it]

Processing Datasets in fcps:  78%|███████▊  | 7/9 [01:43<00:20, 10.45s/it]

Processing Datasets in fcps:  89%|████████▉ | 8/9 [01:53<00:10, 10.25s/it]

Processing Datasets in fcps: 100%|██████████| 9/9 [02:06<00:00, 11.27s/it]

                                                                          
Processing Batteries:  50%|█████     | 1/2 [02:06<02:06, 126.77s/it]

Processing Datasets in uci:   0%|         

In [21]:
df_tsne

,Genie+TSNE15 NCA Score,Genie+TSNE30 NCA Score,Genie+TSNE45 NCA Score
0,0.947500,0.947500,0.947500
1,0.640000,0.640000,0.640000
2,0.691593,0.691593,0.691593
3,1.000000,1.000000,1.000000
4,1.000000,1.000000,1.000000
5,1.000000,1.000000,1.000000
6,0.646667,0.646667,0.646667
7,0.997500,0.997500,0.997500
8,1.000000,1.000000,1.000000
9,0.315563,0.311317,0.319559


In [22]:
umap_neighbors = [2,10,20,50]

scores_lists = {}
for neighbors in tqdm.tqdm(umap_neighbors, desc="Processing UMAP Neighbors"):
  for battery in tqdm.tqdm(battery_datasets_dict.keys(), desc="Processing Batteries"):
    for dataset in tqdm.tqdm(battery_datasets_dict[battery], desc=f"Processing Datasets in {battery}", leave=False):
      column_name = 'Genie+UMAP'+ str(neighbors) +' NCA Score'
      if column_name not in scores_lists:
        scores_lists[column_name] = []
      scores_lists[column_name].append(get_scores_with_umap(battery, dataset, n_neighbors=neighbors))

df_umap = pd.DataFrame.from_dict(scores_lists)

Processing Batteries:   0%|          | 0/2 [00:00<?, ?it/s]

Processing Datasets in fcps:   0%|          | 0/9 [00:00<?, ?it/s]

Processing Datasets in fcps:  11%|█         | 1/9 [00:11<01:35, 11.96s/it]

Processing Datasets in fcps:  22%|██▏       | 2/9 [00:14<00:46,  6.66s/it]/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Processing Datasets in fcps:  33%|███▎      | 3/9 [11:22<30:51, 308.65s/it]

Processing Datasets in fcps:  44%|████▍     | 4/9 [11:23<15:35, 187.18s/it]/usr/local/lib/python3.12/dist-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


Processing Datasets in fcps:  56%|█████▌    | 5/9 [11:24<07:59, 119.99s/it]

Processing Datas

In [23]:
df_umap

,Genie+UMAP2 NCA Score,Genie+UMAP10 NCA Score,Genie+UMAP20 NCA Score,Genie+UMAP50 NCA Score
0,0.267500,0.945000,0.937500,0.940000
1,0.276000,0.500000,1.000000,0.500000
2,0.040599,0.614918,0.922777,0.947721
3,0.262500,1.000000,1.000000,1.000000
4,0.440000,1.000000,1.000000,1.000000
5,0.448160,1.000000,1.000000,1.000000
6,0.230000,0.550000,0.680000,0.783333
7,0.572500,0.997500,0.997500,0.997500
8,0.407480,1.000000,1.000000,1.000000
9,0.278843,0.374418,0.214971,0.251520


In [24]:
df = pd.concat([df, df_tsne, df_umap], axis=1)
df

,Battery,Dataset,Genie NCA Score,Genie+TSNE15 NCA Score,Genie+TSNE30 NCA Score,Genie+TSNE45 NCA Score,Genie+UMAP2 NCA Score,Genie+UMAP10 NCA Score,Genie+UMAP20 NCA Score,Genie+UMAP50 NCA Score
0,fcps,atom,0.977500,0.947500,0.947500,0.947500,0.267500,0.945000,0.937500,0.940000
1,fcps,chainlink,1.000000,0.640000,0.640000,0.640000,0.276000,0.500000,1.000000,0.500000
2,fcps,engytime,0.918870,0.691593,0.691593,0.691593,0.040599,0.614918,0.922777,0.947721
3,fcps,hepta,1.000000,1.000000,1.000000,1.000000,0.262500,1.000000,1.000000,1.000000
4,fcps,lsun,1.000000,1.000000,1.000000,1.000000,0.440000,1.000000,1.000000,1.000000
5,fcps,target,1.000000,1.000000,1.000000,1.000000,0.448160,1.000000,1.000000,1.000000
6,fcps,tetra,0.736667,0.646667,0.646667,0.646667,0.230000,0.550000,0.680000,0.783333
7,fcps,twodiamonds,0.987500,0.997500,0.997500,0.997500,0.572500,0.997500,0.997500,0.997500
8,fcps,wingnut,1.000000,1.000000,1.000000,1.000000,0.407480,1.000000,1.000000,1.000000
9,uci,ecoli,0.405951,0.315563,0.311317,0.319559,0.278843,0.374418,0.214971,0.251520


In [26]:
cols_to_display = ['Battery','Dataset','Genie NCA Score',	'Genie+TSNE15 NCA Score',	'Genie+TSNE30 NCA Score',	'Genie+TSNE45 NCA Score', 'Genie+UMAP2 NCA Score',	'Genie+UMAP10 NCA Score',	'Genie+UMAP20 NCA Score',	'Genie+UMAP50 NCA Score']
numerical_cols = ['Genie NCA Score',	'Genie+TSNE15 NCA Score',	'Genie+TSNE30 NCA Score',	'Genie+TSNE45 NCA Score', 'Genie+UMAP2 NCA Score',	'Genie+UMAP10 NCA Score',	'Genie+UMAP20 NCA Score',	'Genie+UMAP50 NCA Score']
df[cols_to_display].style.highlight_max(axis=1, subset=numerical_cols)

,Battery,Dataset,Genie NCA Score,Genie+TSNE15 NCA Score,Genie+TSNE30 NCA Score,Genie+TSNE45 NCA Score,Genie+UMAP2 NCA Score,Genie+UMAP10 NCA Score,Genie+UMAP20 NCA Score,Genie+UMAP50 NCA Score
0,fcps,atom,0.977500,0.947500,0.947500,0.947500,0.267500,0.945000,0.937500,0.940000
1,fcps,chainlink,1.000000,0.640000,0.640000,0.640000,0.276000,0.500000,1.000000,0.500000
2,fcps,engytime,0.918870,0.691593,0.691593,0.691593,0.040599,0.614918,0.922777,0.947721
3,fcps,hepta,1.000000,1.000000,1.000000,1.000000,0.262500,1.000000,1.000000,1.000000
4,fcps,lsun,1.000000,1.000000,1.000000,1.000000,0.440000,1.000000,1.000000,1.000000
5,fcps,target,1.000000,1.000000,1.000000,1.000000,0.448160,1.000000,1.000000,1.000000
6,fcps,tetra,0.736667,0.646667,0.646667,0.646667,0.230000,0.550000,0.680000,0.783333
7,fcps,twodiamonds,0.987500,0.997500,0.997500,0.997500,0.572500,0.997500,0.997500,0.997500
8,fcps,wingnut,1.000000,1.000000,1.000000,1.000000,0.407480,1.000000,1.000000,1.000000
9,uci,ecoli,0.405951,0.315563,0.311317,0.319559,0.278843,0.374418,0.214971,0.251520
